# Chapter 5 — R-601 Mechanical Design
Calculates the R-601 mechanical dimensions, pressure thicknesses, loads and insulation.

## 5.1 Imports
Loads simple numerical and table libraries used throughout the mechanical calculations.

In [1]:
import math
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 5.2 Final process basis and mechanical assumptions
Defines the reactor process conditions and mechanical design values.

In [2]:
PROCESS = {
    "tube_count": 1757, "tube_id_m": 0.040, "tube_wall_m": 0.00198, "bed_length_m": 8.955905,
    "catalyst_mass_kg": 23728.612, "reactor_inlet_bar_a": 35.984, "reactor_inlet_C": 198.53,
    "hotspot_C": 272.04, "coolant_bar_a": 17.107, "reactor_dP_bar": 0.899,
    "reactor_heat_W": 5.965e6, "ambient_C": 25.0
}
DESIGN = {
    "atmosphere_bar": 1.01325, "gas_design_bar_g": 40.0, "shell_design_bar_g": 20.0,
    "design_temperature_C": 300.0, "shell_stress_MPa": 120.0, "gas_stress_MPa": 115.0,
    "tube_stress_MPa": 115.0, "shell_weld_E": 0.85, "gas_weld_E": 0.85,
    "shell_CA_mm": 2.0, "gas_CA_mm": 1.5, "tube_CA_mm": 0.0,
    "shell_nominal_mm": 30.0, "channel_nominal_mm": 60.0, "head_nominal_mm": 60.0,
    "tube_mill_under_tolerance": 0.125, "steel_density_kg_m3": 7850.0,
    "shell_ID_m": 2.60, "channel_ID_m": 2.60, "tube_pitch_ratio": 1.25,
    "head_count": 2, "channel_count": 2, "channel_length_each_m": 0.50,
    "shell_extra_length_m": 0.50, "tubesheet_count": 2, "tubesheet_t_mm": 120.0,
    "support_plate_count": 5, "support_plate_t_mm": 20.0,
    "steel_alpha_per_K": 12e-6, "crmo_alpha_per_K": 13e-6,
    "shell_mean_operating_C": 205.0, "tube_mean_operating_C": 240.0,
    "water_density_kg_m3": 850.0, "water_fill_fraction": 0.75,
    "other_internals_kg": 2500.0, "gas_kg": 1000.0, "nozzle_mass_allowance_kg": 1500.0,
    "insulation_density_kg_m3": 100.0, "cladding_areal_mass_kg_m2": 3.0,
    "skirt_length_m": 1.80, "skirt_thickness_mm": 20.0,
    "skirt_allowable_MPa": 120.0, "skirt_load_factor": 1.5,
    "insulation_k_W_mK": 0.060, "insulation_thickness_m": 0.075,
    "external_h_W_m2K": 8.0, "external_radiation_h_W_m2K": 5.0,
    "insulation_outer_limit_C": 60.0,
    "water_side_hydrotest_stress_ratio": None, "gas_side_hydrotest_stress_ratio": None,
    "SANS347_category": None, "tube_external_pressure_allowable_bar": None,
    "UHX_tubesheet_acceptance": None, "API941_material_acceptance": None
}

assert PROCESS["tube_count"] > 0 and PROCESS["bed_length_m"] > 0
assert DESIGN["shell_ID_m"] > PROCESS["tube_id_m"]
print("Normal gas pressure (bar(g)):", round(PROCESS["reactor_inlet_bar_a"] - DESIGN["atmosphere_bar"], 3))
print("Normal cooling pressure (bar(g)):", round(PROCESS["coolant_bar_a"] - DESIGN["atmosphere_bar"], 3))

Normal gas pressure (bar(g)): 34.971
Normal cooling pressure (bar(g)): 16.094


## 5.3 Tube layout, bundle size and heat-transfer surface
Calculates tube pitch, bundle diameter, shell clearance, catalyst volume and tube area.

The tube geometry and bundle diameter are calculated using (Sinnott, 2005):

$$
d_o=d_i+2t_w,\qquad p_t=1.25d_o
$$

$$
D_b=d_o\left(\frac{N_t}{K_1}\right)^{1/n_1},\qquad K_1=0.319,\quad n_1=2.142
$$

$$
A_o=N_t\pi d_oL,\qquad V_b=\frac{N_t\pi d_i^2L}{4}
$$

In [3]:
N = PROCESS["tube_count"]
di = PROCESS["tube_id_m"]
tw = PROCESS["tube_wall_m"]
L = PROCESS["bed_length_m"]
do = di + 2*tw
pitch = DESIGN["tube_pitch_ratio"]*do
Db = do*(N/0.319)**(1/2.142)
Ds = DESIGN["shell_ID_m"]
Dchannel = DESIGN["channel_ID_m"]
clearance_mm = 1000*(Ds - Db)/2
A_tubes_m2 = N*math.pi*do*L
V_bed_m3 = N*math.pi*di**2*L/4
A_open_tubes_m2 = N*math.pi*di**2/4
A_water_free_m2 = math.pi*(Ds**2 - N*do**2)/4
if clearance_mm < 0 or A_water_free_m2 <= 0: raise ValueError("Tube bundle does not fit the shell")
print(f"Tube OD = {do*1000:.2f} mm | pitch = {pitch*1000:.2f} mm")
print(f"Bundle diameter = {Db:.3f} m | radial clearance = {clearance_mm:.1f} mm")
print(f"Tube outside area = {A_tubes_m2:.1f} m² | packed volume = {V_bed_m3:.2f} m³")

Tube OD = 43.96 mm | pitch = 54.95 mm
Bundle diameter = 2.452 m | radial clearance = 73.9 mm
Tube outside area = 2173.1 m² | packed volume = 19.77 m³


## 5.4 Cylindrical-shell and gas-channel thickness
Calculates shell and channel wall thicknesses using ASME VIII-1, UG-27 (ASME, 2025).

$$
t_c=\frac{PR}{SE-0.6P},\qquad t_l=\frac{PR}{2SE+0.4P}
$$

$$
t_{req}=\max(t_c,t_l)+CA,\qquad t_{available}=t_{nom}-CA
$$

In [4]:
def cylinder_thickness_mm(P_MPa, R_mm, S_MPa, E, CA_mm):
    if S_MPa*E <= 0.6*P_MPa: raise ValueError("Pressure exceeds membrane formula validity")
    t_c = P_MPa*R_mm/(S_MPa*E - 0.6*P_MPa)
    t_l = P_MPa*R_mm/(2*S_MPa*E + 0.4*P_MPa)
    return {"hoop_mm": t_c, "axial_mm": t_l, "required_mm": max(t_c, t_l) + CA_mm}

shell_calc = cylinder_thickness_mm(DESIGN["shell_design_bar_g"]/10, Ds*500, DESIGN["shell_stress_MPa"], DESIGN["shell_weld_E"], DESIGN["shell_CA_mm"])
gas_calc = cylinder_thickness_mm(DESIGN["gas_design_bar_g"]/10, Dchannel*500, DESIGN["gas_stress_MPa"], DESIGN["gas_weld_E"], DESIGN["gas_CA_mm"])
print("Water shell:", shell_calc, "nominal passes screening:", DESIGN["shell_nominal_mm"] >= shell_calc["required_mm"])
print("Gas channel:", gas_calc, "nominal passes screening:", DESIGN["channel_nominal_mm"] >= gas_calc["required_mm"])

Water shell: {'hoop_mm': 25.793650793650794, 'axial_mm': 12.6953125, 'required_mm': 27.793650793650794} nominal passes screening: True
Gas channel: {'hoop_mm': 54.53592029365496, 'axial_mm': 26.382546930492136, 'required_mm': 56.03592029365496} nominal passes screening: True


## 5.5 Ellipsoidal gas heads
Calculates the wall thickness and depth of 2:1 ellipsoidal heads (ASME VIII-1, UG-32(d); ASME, 2025).

$$
t_h=\frac{PD}{2SE-0.2P}+CA,\qquad h=\frac{D}{4}
$$

In [5]:
def ellipsoidal_head_thickness_mm(P_MPa, D_mm, S_MPa, E, CA_mm):
    if 2*S_MPa*E <= 0.2*P_MPa: raise ValueError("Invalid head pressure denominator")
    return P_MPa*D_mm/(2*S_MPa*E - 0.2*P_MPa) + CA_mm

head_t_req = ellipsoidal_head_thickness_mm(DESIGN["gas_design_bar_g"]/10, Dchannel*1000, DESIGN["gas_stress_MPa"], DESIGN["gas_weld_E"], DESIGN["gas_CA_mm"])
head_depth_m = Dchannel/4
print(f"Required head thickness = {head_t_req:.2f} mm")
print(f"Selected nominal head = {DESIGN['head_nominal_mm']:.1f} mm | depth = {head_depth_m:.3f} m")

Required head thickness = 54.92 mm
Selected nominal head = 60.0 mm | depth = 0.650 m


## 5.6 Catalyst-tube internal and external pressure
Calculates the tube wall requirement and pressure difference (ASME VIII-1, UG-27 and UG-28; ASME, 2025).

$$
t_{tube}=\frac{P(d_i/2)}{S-0.6P}+CA,\qquad t_{min,delivered}=t_{nom}(1-f_{mill})
$$

$$
\Delta P_{reverse}=P_{shell,design}-P_{gas,min}
$$

In [6]:
tube_calc = cylinder_thickness_mm(DESIGN["gas_design_bar_g"]/10, di*500, DESIGN["tube_stress_MPa"], 1.0, DESIGN["tube_CA_mm"])
tube_delivered_mm = tw*1000*(1 - DESIGN["tube_mill_under_tolerance"])
tube_internal_ok = tube_delivered_mm >= tube_calc["required_mm"]
reverse_dP_bar = DESIGN["shell_design_bar_g"]
external_allowable = DESIGN["tube_external_pressure_allowable_bar"]
tube_external_ok = None if external_allowable is None else external_allowable >= reverse_dP_bar
print(f"Tube internal-pressure minimum = {tube_calc['required_mm']:.3f} mm | delivered = {tube_delivered_mm:.3f} mm | pass = {tube_internal_ok}")
print(f"Reverse design differential = {reverse_dP_bar:.2f} bar | external-collapse pass = {tube_external_ok}")

Tube internal-pressure minimum = 0.710 mm | delivered = 1.732 mm | pass = True
Reverse design differential = 20.00 bar | external-collapse pass = None


## 5.7 Preliminary tubesheet and support arrangement
Calculates preliminary tube and tubesheet pressure forces (ASME VIII-1, UHX; ASME, 2025).

$$
F_{tube}=\Delta P\frac{\pi d_i^2}{4},\qquad F_{gross}=\Delta P\frac{\pi D_{sheet}^2}{4}
$$

In [7]:
dP_normal_Pa = max(0, DESIGN["gas_design_bar_g"] - DESIGN["shell_design_bar_g"])*1e5
F_tube_N = dP_normal_Pa*math.pi*di**2/4
F_sheet_gross_N = dP_normal_Pa*math.pi*Dchannel**2/4
support_spacing_m = L/(DESIGN["support_plate_count"] + 1)
print(f"Tube pressure-force screening = {F_tube_N/1000:.2f} kN per tube")
print(f"Gross tubesheet differential-force indicator = {F_sheet_gross_N/1e6:.2f} MN")
print(f"Initial unsupported tube span = {support_spacing_m:.2f} m")
print("ASME UHX tubesheet verification:", DESIGN["UHX_tubesheet_acceptance"])

Tube pressure-force screening = 2.51 kN per tube
Gross tubesheet differential-force indicator = 10.62 MN
Initial unsupported tube span = 1.49 m
ASME UHX tubesheet verification: None


## 5.8 Differential thermal expansion
Calculates the difference in expansion between the catalyst tubes and shell (Young and Budynas, *Roark’s Formulas for Stress and Strain*).

$$
\Delta L_i=\alpha_i L_i(T_{mean,i}-T_{ref})
$$

$$
\Delta L_{relative}=\Delta L_{tube}-\Delta L_{shell}
$$

In [8]:
Tref = PROCESS["ambient_C"]
dL_tube_m = DESIGN["crmo_alpha_per_K"]*L*(DESIGN["tube_mean_operating_C"] - Tref)
dL_shell_m = DESIGN["steel_alpha_per_K"]*L*(DESIGN["shell_mean_operating_C"] - Tref)
dL_relative_mm = 1000*(dL_tube_m - dL_shell_m)
print(f"Tube expansion = {dL_tube_m*1000:.2f} mm | shell expansion = {dL_shell_m*1000:.2f} mm")
print(f"Relative expansion = {dL_relative_mm:+.2f} mm (screening only)")

Tube expansion = 25.03 mm | shell expansion = 19.34 mm
Relative expansion = +5.69 mm (screening only)


## 5.9 Vessel material quantities and weights
Calculates the component metal volumes, masses and operating weights using standard geometric relationships.

$$
V_{cylinder}=\frac{\pi L}{4}(D_o^2-D_i^2),\qquad m=\rho V
$$

The ellipsoidal-head surface area is calculated from:

$$
r(z)=a\sqrt{1-(z/b)^2},\qquad A=2\pi\int_0^b r(z)\sqrt{1+[dr/dz]^2}\,dz
$$

In [9]:
def cylinder_metal_volume(Din_m, t_m, length_m):
    return math.pi*((Din_m + 2*t_m)**2 - Din_m**2)*length_m/4

def ellipsoid_area_m2(D_m):
    a, b = D_m/2, D_m/4
    theta = np.linspace(0, math.pi/2, 5001)
    ds = np.sqrt((a*np.cos(theta))**2 + (b*np.sin(theta))**2)
    return float(2*math.pi*np.trapezoid(a*np.sin(theta)*ds, theta))

rho = DESIGN["steel_density_kg_m3"]
Ls = L + DESIGN["shell_extra_length_m"]
t_shell = DESIGN["shell_nominal_mm"]/1000
t_channel = DESIGN["channel_nominal_mm"]/1000
t_head = DESIGN["head_nominal_mm"]/1000
m_shell = rho*cylinder_metal_volume(Ds, t_shell, Ls)
m_tubes = N*rho*math.pi*(do**2 - di**2)*L/4
m_channel = DESIGN["channel_count"]*rho*cylinder_metal_volume(Dchannel, t_channel, DESIGN["channel_length_each_m"])
head_area = ellipsoid_area_m2(Dchannel)
m_heads = DESIGN["head_count"]*head_area*t_head*rho
m_tubesheets = DESIGN["tubesheet_count"]*rho*DESIGN["tubesheet_t_mm"]/1000*A_water_free_m2
m_supports = DESIGN["support_plate_count"]*rho*DESIGN["support_plate_t_mm"]/1000*A_water_free_m2
V_water = A_water_free_m2*L*DESIGN["water_fill_fraction"]
m_water = V_water*DESIGN["water_density_kg_m3"]
m_full_hydro_water = A_water_free_m2*Ls*1000
m_skirt = rho*cylinder_metal_volume(Ds, DESIGN["skirt_thickness_mm"]/1000, DESIGN["skirt_length_m"])
print(pd.Series({"Shell kg":m_shell,"Tubes kg":m_tubes,"Gas channels kg":m_channel,"Gas heads kg":m_heads,"Tubesheets kg (estimated)":m_tubesheets,"Tube supports kg (estimated)":m_supports,"Skirt kg":m_skirt,"Operating water kg":m_water,"Full hydrotest shell water kg":m_full_hydro_water}).round(1).to_string())

Shell kg                        18,399.200
Tubes kg                        32,255.800
Gas channels kg                  3,936.000
Gas heads kg                     6,902.700
Tubesheets kg (estimated)        4,978.600
Tube supports kg (estimated)     2,074.400
Skirt kg                         2,326.100
Operating water kg              15,087.500
Full hydrotest shell water kg   24,987.900


## 5.10 Skirt axial-load calculation
Calculates the skirt axial stress from the vessel weight (Chattopadhyay, 2005).

$$
W=mg,\qquad A_{sk}=\frac{\pi}{4}[(D+2t_{sk})^2-D^2]
$$

$$
\sigma_z=\frac{f_LW}{A_{sk}}
$$

In [10]:
m_fixed = m_shell + m_tubes + m_channel + m_heads + m_tubesheets + m_supports + m_skirt + DESIGN["other_internals_kg"] + DESIGN["nozzle_mass_allowance_kg"]
m_operating = m_fixed + PROCESS["catalyst_mass_kg"] + m_water + DESIGN["gas_kg"]
m_hydro_screen = m_fixed + PROCESS["catalyst_mass_kg"] + m_full_hydro_water
A_skirt = math.pi*((Ds + 2*DESIGN["skirt_thickness_mm"]/1000)**2 - Ds**2)/4
sigma_skirt_MPa = DESIGN["skirt_load_factor"]*max(m_operating, m_hydro_screen)*9.81/A_skirt/1e6
skirt_axial_ok = sigma_skirt_MPa <= DESIGN["skirt_allowable_MPa"]
print(f"Operating mass ~ {m_operating/1000:.1f} t | provisional hydrotest mass ~ {m_hydro_screen/1000:.1f} t")
print(f"Skirt axial stress with factor = {sigma_skirt_MPa:.2f} MPa | axial screening pass = {skirt_axial_ok}")

Operating mass ~ 114.7 t | provisional hydrotest mass ~ 123.6 t
Skirt axial stress with factor = 11.05 MPa | axial screening pass = True


## 5.11 Insulation surface temperature and heat loss
Calculates insulation heat loss and surface temperature (Incropera et al.; ISO 12241).

$$
R_{cond}=\frac{\ln(r_2/r_1)}{2\pi kL},\qquad R_{out}=\frac{1}{2\pi r_2L(h_c+h_r)}
$$

$$
Q=\frac{T_{wall}-T_{amb}}{R_{cond}+R_{out}},\qquad T_{surface}=T_{amb}+QR_{out}
$$

In [11]:
def insulation_result(thickness_m):
    r1 = (Ds + 2*t_shell)/2
    r2 = r1 + thickness_m
    k = DESIGN["insulation_k_W_mK"]
    h = DESIGN["external_h_W_m2K"] + DESIGN["external_radiation_h_W_m2K"]
    Rcond = math.log(r2/r1)/(2*math.pi*k*Ls) if thickness_m > 0 else 0
    Rout = 1/(2*math.pi*r2*Ls*h)
    Q = (DESIGN["shell_mean_operating_C"] - PROCESS["ambient_C"])/(Rcond + Rout)
    Ts = PROCESS["ambient_C"] + Q*Rout
    return Q, Ts

ins_t = DESIGN["insulation_thickness_m"]
ins_Q_W, ins_Ts_C = insulation_result(ins_t)
ins_area_m2 = 2*math.pi*(Ds/2+t_shell+ins_t)*Ls
ins_mass_kg = math.pi*((Ds+2*t_shell+2*ins_t)**2 - (Ds+2*t_shell)**2)*Ls*DESIGN["insulation_density_kg_m3"]/4
cladding_mass_kg = ins_area_m2*DESIGN["cladding_areal_mass_kg_m2"]
required_ins_m = None
for candidate_mm in range(0, 301, 5):
    if insulation_result(candidate_mm/1000)[1] <= DESIGN["insulation_outer_limit_C"]:
        required_ins_m = candidate_mm/1000
        break
print(f"Selected insulation = {ins_t*1000:.0f} mm | surface = {ins_Ts_C:.1f} °C | cylindrical heat loss = {ins_Q_W/1000:.1f} kW")
print("Minimum trial insulation for surface criterion:", None if required_ins_m is None else f"{required_ins_m*1000:.0f} mm")
print(f"Insulation + cladding mass (screening) = {(ins_mass_kg + cladding_mass_kg)/1000:.2f} t")

Selected insulation = 75 mm | surface = 35.2 °C | cylindrical heat loss = 11.0 kW
Minimum trial insulation for surface criterion: 20 mm
Insulation + cladding mass (screening) = 0.86 t


## 5.12 Pressure capacity and hydrotest screening
Calculates pressure capacities and the hydrotest screening pressure (ASME VIII-1, UG-27, UG-32 and UG-99; ASME, 2025).

$$
P_{cyl}=\frac{SEt}{R+0.6t},\qquad P_{head}=\frac{2SEt}{D+0.2t}
$$

$$
P_{test}=1.3\,MAWP\frac{S_{test}}{S_{design}}
$$

In [12]:
def pressure_capacity_cylinder_bar(t_mm, CA_mm, R_mm, S_MPa, E):
    te = t_mm - CA_mm
    return 10*S_MPa*E*te/(R_mm + 0.6*te) if te > 0 else 0

def pressure_capacity_head_bar(t_mm, CA_mm, D_mm, S_MPa, E):
    te = t_mm - CA_mm
    return 10*2*S_MPa*E*te/(D_mm + 0.2*te) if te > 0 else 0

shell_membrane_MAWP = pressure_capacity_cylinder_bar(DESIGN["shell_nominal_mm"], DESIGN["shell_CA_mm"], Ds*500, DESIGN["shell_stress_MPa"], DESIGN["shell_weld_E"])
channel_membrane_MAWP = pressure_capacity_cylinder_bar(DESIGN["channel_nominal_mm"], DESIGN["gas_CA_mm"], Dchannel*500, DESIGN["gas_stress_MPa"], DESIGN["gas_weld_E"])
head_membrane_MAWP = pressure_capacity_head_bar(DESIGN["head_nominal_mm"], DESIGN["gas_CA_mm"], Dchannel*1000, DESIGN["gas_stress_MPa"], DESIGN["gas_weld_E"])
gas_membrane_MAWP = min(channel_membrane_MAWP, head_membrane_MAWP)
test_water = None if DESIGN["water_side_hydrotest_stress_ratio"] is None else 1.3*shell_membrane_MAWP*DESIGN["water_side_hydrotest_stress_ratio"]
test_gas = None if DESIGN["gas_side_hydrotest_stress_ratio"] is None else 1.3*gas_membrane_MAWP*DESIGN["gas_side_hydrotest_stress_ratio"]
print(f"Water-side membrane pressure capacity = {shell_membrane_MAWP:.2f} bar(g)")
print(f"Gas-side membrane pressure capacity = {gas_membrane_MAWP:.2f} bar(g)")
print("Water hydrotest code verification input required; calculated tentative value:", test_water)
print("Gas hydrotest code verification input required; calculated tentative value:", test_gas)

Water-side membrane pressure capacity = 21.69 bar(g)
Gas-side membrane pressure capacity = 42.83 bar(g)
Water hydrotest code verification input required; calculated tentative value: None
Gas hydrotest code verification input required; calculated tentative value: None


## 5.13 Design verification
Displays the mechanical calculation checks and their results.

In [13]:
CHECKS = {
    "Tube bundle fits selected shell": clearance_mm > 0 and A_water_free_m2 > 0,
    "Water-shell internal membrane thickness": DESIGN["shell_nominal_mm"] >= shell_calc["required_mm"],
    "Gas-channel internal membrane thickness": DESIGN["channel_nominal_mm"] >= gas_calc["required_mm"],
    "Gas-head internal membrane thickness": DESIGN["head_nominal_mm"] >= head_t_req,
    "Tube internal-pressure screening incl. mill tolerance": tube_internal_ok,
    "Insulation temperature screening": ins_Ts_C <= DESIGN["insulation_outer_limit_C"],
    "Skirt direct axial stress screening": skirt_axial_ok,
    "Tube external collapse ASME UG-28": tube_external_ok,
    "Tubesheet ASME UHX": DESIGN["UHX_tubesheet_acceptance"],
    "Hydrogen service API RP 941": DESIGN["API941_material_acceptance"],
    "SANS 347 classification completed": None if DESIGN["SANS347_category"] is None else True,
    "Hydrotest procedure inputs completed": None if test_water is None or test_gas is None else True,
    "Wind/seismic/nozzle/support local loads": None,
    "Nozzle reinforcement and relief-system checks": None
}
check_table = pd.DataFrame([{"Check": k, "Status": "NOT VERIFIED" if v is None else ("PASS - screening" if v else "FAIL")} for k, v in CHECKS.items()])
print(check_table.to_string(index=False))
print("\nNOTE: PASS - screening is NOT certified code compliance.")

                                                Check           Status
                      Tube bundle fits selected shell PASS - screening
              Water-shell internal membrane thickness PASS - screening
              Gas-channel internal membrane thickness PASS - screening
                 Gas-head internal membrane thickness PASS - screening
Tube internal-pressure screening incl. mill tolerance PASS - screening
                     Insulation temperature screening PASS - screening
                  Skirt direct axial stress screening PASS - screening
                    Tube external collapse ASME UG-28     NOT VERIFIED
                                   Tubesheet ASME UHX     NOT VERIFIED
                          Hydrogen service API RP 941     NOT VERIFIED
                    SANS 347 classification completed     NOT VERIFIED
                 Hydrotest procedure inputs completed     NOT VERIFIED
              Wind/seismic/nozzle/support local loads     NOT VERIFIED
      

## 5.14 Mechanical results table
Displays the final mechanical results and calculation checks.

In [14]:
RESULTS = [
    ("Number of tubes", N, "-"), ("Tube ID", di*1000, "mm"), ("Tube OD", do*1000, "mm"),
    ("Tube wall", tw*1000, "mm"), ("Packed length", L, "m"), ("Tube pitch", pitch*1000, "mm"),
    ("Bundle diameter", Db, "m"), ("Shell ID", Ds, "m"), ("Radial clearance", clearance_mm, "mm"),
    ("Tube heat transfer area", A_tubes_m2, "m2"), ("Catalyst packed volume", V_bed_m3, "m3"),
    ("Water shell required thickness", shell_calc["required_mm"], "mm"), ("Water shell nominal", DESIGN["shell_nominal_mm"], "mm"),
    ("Gas channel required thickness", gas_calc["required_mm"], "mm"), ("Gas channel nominal", DESIGN["channel_nominal_mm"], "mm"),
    ("Gas head required thickness", head_t_req, "mm"), ("Gas head nominal", DESIGN["head_nominal_mm"], "mm"),
    ("Tube required internal-pressure thickness", tube_calc["required_mm"], "mm"),
    ("Relative thermal expansion", dL_relative_mm, "mm"), ("Operating mass estimate", m_operating/1000, "t"),
    ("Hydrotest mass estimate", m_hydro_screen/1000, "t"), ("Skirt direct axial screening stress", sigma_skirt_MPa, "MPa"),
    ("Insulation thickness", ins_t*1000, "mm"), ("Insulation outer surface temperature", ins_Ts_C, "deg C"),
    ("Insulation shell heat loss", ins_Q_W/1000, "kW"), ("Reactor cooling duty", PROCESS["reactor_heat_W"]/1e6, "MW")
]
results_table = pd.DataFrame(RESULTS, columns=["Parameter", "Value", "Unit"])
print("\nMECHANICAL DESIGN RESULTS")
print(results_table.to_string(index=False))
print("\nDESIGN VERIFICATION REGISTER")
print(check_table.to_string(index=False))


MECHANICAL DESIGN RESULTS
                                Parameter     Value  Unit
                          Number of tubes 1,757.000     -
                                  Tube ID    40.000    mm
                                  Tube OD    43.960    mm
                                Tube wall     1.980    mm
                            Packed length     8.956     m
                               Tube pitch    54.950    mm
                          Bundle diameter     2.452     m
                                 Shell ID     2.600     m
                         Radial clearance    73.924    mm
                  Tube heat transfer area 2,173.145    m2
                   Catalyst packed volume    19.774    m3
           Water shell required thickness    27.794    mm
                      Water shell nominal    30.000    mm
           Gas channel required thickness    56.036    mm
                      Gas channel nominal    60.000    mm
              Gas head required thickness    

## 5.15 References
1. ASME (2025). *ASME Boiler and Pressure Vessel Code, Section VIII, Division 1.* UG-27 (cylindrical shells), UG-28 (external pressure), UG-32 (formed heads), UG-37 (openings), UG-99 (hydrostatic test), UHX (tubesheets). American Society of Mechanical Engineers. https://www.asme.org/publications-submissions/books/find-book/guidebook-design-asme-section-viii-pressure-vessels-fourth-edition
2. ASME (2025). *ASME BPVC Section II, Part D — Properties.* Allowable stresses and external-pressure chart data. Verify values against the licensed edition.
3. Sinnott, R. K. (2005). *Chemical Engineering Design*, 4th ed. Tube pitch and bundle-diameter correlation. See reproduced constants in https://www.sciencedirect.com/topics/engineering/baffle-cut
4. Tubular Exchanger Manufacturers Association (TEMA) (2026). *Standards of the Tubular Exchanger Manufacturers Association.* Tube layout, supports and fabrication practice. https://tema.org/standards/
5. API. *API Recommended Practice 941: Steels for Hydrogen Service at Elevated Temperatures and Pressures in Petroleum Refineries and Petrochemical Plants.* Material screening for high-temperature hydrogen attack. https://www.api.org/
6. Chattopadhyay, S. (2005). *Pressure Vessels: Design and Practice.* CRC Press. Vessel-support loads and structural design methodology.
7. Young, W. C. and Budynas, R. G. *Roark's Formulas for Stress and Strain.* Thermal expansion and structural load relationships.
8. Incropera, F. P. et al. *Fundamentals of Heat and Mass Transfer.* Cylindrical conduction and convection thermal resistances.
9. ISO 12241. *Thermal insulation for building equipment and industrial installations — Calculation rules.* Cylindrical insulation sizing, heat loss and surface temperatures.
10. South Africa (2009). *Pressure Equipment Regulations* (Occupational Health and Safety Act), with SANS 347:2026 categorisation and conformity assessment. https://www.gov.za/sites/default/files/gcis_document/202602/54218-gen3800.pdf

Model limitations: The 2.60 m shell, selected wall nominal values, materials, temperatures, water fill, mean metal temperatures, support details and insulation properties are transparent preliminary inputs. Tubesheet and external-pressure code checks cannot be deduced from simplified internal-pressure equations. The gas-side material must be verified for actual H₂ partial pressure and temperature. Hydrotest and code categories must not be reported as approved until independently checked.